# Significance testing
This notebook uses ANOVA (means) and bootstrapping (extremes) to calculate if groups are significantly different from one another.
Because certain configurations may be meaningless, these results should only be used qualitatively.

In [1]:
import xarray
import pickle
import statsmodels.api as sm
from statsmodels.formula.api import ols
import statsmodels.formula.api as smf
import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import numpy as np
from itertools import combinations

In [6]:
# Grab each subregion individually and save for significance testing

df = pickle.load(open('/glade/work/tcorrie/ARdata/pmean/prec_mean_all.pkl', 'rb'))
wshed_list = ['1503', '1605', '1606', '1701', '1702', '1703', '1705', '1706', '1707', '1708', '1709', '1710', '1711', '1712', '1801', '1802', '1803', '1804', '1805', '1806', '1807', '1808', '1809', '1810']
for wshed in wshed_list:
    df_huc4 = df[wshed]
    print(wshed, end='\t')
    rows = []
    
    for era in ['obs', 'historical', 'near-term', 'end-of-century']:
        for ARDT in ['G18', 'TE', 'tARget']:
            for model in ['wrfera5', 'prism', 'cmip6nonbc', 'cmip6bc', 'cesm2le']:
                if era=='obs':
                    key3 = 'historical'
                else:
                    key3 = era
                if era == 'obs' and model not in ['wrfera5', 'prism']:
                    continue
                if era != 'obs' and model in ['wrfera5', 'prism']:
                    continue
                df_huc4_oneconfig = df_huc4[model][ARDT][key3]
                for item in df_huc4_oneconfig:
                    rows.append((item, era, ARDT, model)) # Convert dictionary into a pandas table to remove the nesting.
    precip = pd.DataFrame(rows, columns=['precipitation', 'era', 'ARDT', 'model'])
    precip.to_csv(f'intermediate_numpy_files/precip_for_sigtest_{wshed}.csv', index=False)
    print(" saved.")


1503	 saved.
1605	 saved.
1606	 saved.
1701	 saved.
1702	 saved.
1703	 saved.
1705	 saved.
1706	 saved.
1707	 saved.
1708	 saved.
1709	 saved.
1710	 saved.
1711	 saved.
1712	 saved.
1801	 saved.
1802	 saved.
1803	 saved.
1804	 saved.
1805	 saved.
1806	 saved.
1807	 saved.
1808	 saved.
1809	 saved.
1810	 saved.


In [3]:
#precip.to_csv('intermediate_numpy_files/precip_for_anova_test.csv', index=False)
#precipitation = pd.read_csv('intermediate_numpy_files/precip_for_anova_test.csv')
#precip = pd.read_csv('intermediate_numpy_files/precip_for_anova_test.csv')
precip = pd.read_csv('intermediate_numpy_files/precip_for_sigtest_1503.csv') # Change 1503 to whatever subregion ID you need.

In [4]:
# ANOVA and Tukey HSD for one subregion only.

model = ols('precipitation ~ C(era) + C(ARDT) + C(model)', data=precip).fit()
anova_result = sm.stats.anova_lm(model, type=2)
print(anova_result)
print("\n" + "=" * 55)
print("ONE-WAY ANOVA (per categorical variable)")
print("=" * 55)
cat_cols=['era', 'ARDT', 'model']
for cat in cat_cols:
    groups = [group['precipitation'].values for _, group in precip.groupby(cat)]
    f_stat, p_val = stats.f_oneway(*groups)
    print(f"\n{cat}:  F = {f_stat:.4f},  p = {p_val:.4f}",
          "<-- significant" if p_val < 0.05 else "")

print("\n" + "=" * 55)
print("TUKEY HSD (pairwise group comparisons per variable)")
print("=" * 55)

for cat in cat_cols:
    print(f"\n--- {cat} ---")
    tukey = pairwise_tukeyhsd(
        endog  = precip['precipitation'],
        groups = precip[cat],
        alpha  = 0.05,
    )
    print(tukey)

print("\n" + "=" * 55)
print("VARIANCE TESTS (per categorical variable)")
print("=" * 55)

for cat in cat_cols:
    groups = [group['precipitation'].values for _, group in precip.groupby(cat)]

    lev_stat,  lev_p  = stats.levene(*groups)   # robust to non-normality
    bart_stat, bart_p = stats.bartlett(*groups)  # assumes normality

    print(f"\n{cat}:")
    print(f"  Levene:   W = {lev_stat:.4f},  p = {lev_p:.4f}",
          "<-- unequal variances" if lev_p < 0.05 else "")
    print(f"  Bartlett: T = {bart_stat:.4f},  p = {bart_p:.4f}",
          "<-- unequal variances" if bart_p < 0.05 else "")

                df        sum_sq       mean_sq           F    PR(>F)
C(era)         3.0  6.266731e+02    208.891030    3.243772  0.020997
C(ARDT)        2.0  1.108581e+05  55429.029656  860.731826  0.000000
C(model)       4.0  1.020184e+05  25504.592508  396.049049  0.000000
Residual  118297.0  7.618038e+06     64.397560         NaN       NaN

ONE-WAY ANOVA (per categorical variable)

era:  F = 3.1558,  p = 0.0237 <-- significant

ARDT:  F = 845.6557,  p = 0.0000 <-- significant

model:  F = 381.5594,  p = 0.0000 <-- significant

TUKEY HSD (pairwise group comparisons per variable)

--- era ---
      Multiple Comparison of Means - Tukey HSD, FWER=0.05      
    group1       group2   meandiff p-adj   lower  upper  reject
---------------------------------------------------------------
end-of-century historical   0.0813 0.6035 -0.0876 0.2501  False
end-of-century  near-term   0.1971 0.0114  0.0324 0.3618   True
end-of-century        obs   0.0813 0.6035 -0.0876 0.2501  False
    historical 

In [5]:
# This cell is not applicable for this study. Please ignore.

QUANTILE  = 0.99   # <-- change to 0.99 etc.
N_BOOT    = 2000
N_PERM    = 2000
ALPHA     = 0.05
'''
print("=" * 60)
print(f"QUANTILE REGRESSION (q={QUANTILE}) — multi-factor")
print("=" * 60)

formula = 'precipitation ~ C(era) + C(ARDT) + C(model)'
qr_model = smf.quantreg(formula, data=precip).fit(q=QUANTILE)
print(qr_model.summary())

print("\n" + "=" * 60)
print(f"PERMUTATION TEST on {QUANTILE*100:.0f}th percentile (per variable)")
print("=" * 60)

def quantile_stat(groups, q):
    """Range of group quantiles — large value = groups differ."""
    qs = [np.quantile(g, q) for g in groups]
    return max(qs) - min(qs)

for cat in cat_cols:
    grouped   = [g['precipitation'].values for _, g in precip.groupby(cat)]
    observed  = quantile_stat(grouped, QUANTILE)
    pooled    = precip['precipitation'].values

    # Permute labels and recompute stat
    perm_stats = []
    group_sizes = [len(g) for g in grouped]
    for _ in range(N_PERM):
        shuffled = np.random.permutation(pooled)
        perm_groups = []
        idx = 0
        for size in group_sizes:
            perm_groups.append(shuffled[idx:idx+size])
            idx += size
        perm_stats.append(quantile_stat(perm_groups, QUANTILE))

    p_val = np.mean(np.array(perm_stats) >= observed)
    sig   = "<-- significant" if p_val < ALPHA else ""
    print(f"\n{cat}:  observed range = {observed:.4f},  p = {p_val:.4f}  {sig}")

print("\n" + "=" * 60)
print(f"BOOTSTRAP PAIRWISE COMPARISON on {QUANTILE*100:.0f}th percentile")
print("=" * 60)

def boot_quantile_diff(g1, g2, q, n_boot):
    """Bootstrap CI for the difference in q-th percentile between two groups."""
    diffs = []
    for _ in range(n_boot):
        b1 = np.random.choice(g1, size=len(g1), replace=True)
        b2 = np.random.choice(g2, size=len(g2), replace=True)
        diffs.append(np.quantile(b1, q) - np.quantile(b2, q))
    diffs = np.array(diffs)
    ci_lo, ci_hi = np.percentile(diffs, [2.5, 97.5])
    # p-value: proportion of bootstrap diffs on the opposite side of 0
    p = 2 * min(np.mean(diffs >= 0), np.mean(diffs <= 0))
    return np.quantile(g1, q) - np.quantile(g2, q), ci_lo, ci_hi, p

for cat in cat_cols:
    groups     = {name: g['precipitation'].values for name, g in precip.groupby(cat)}
    group_names = list(groups.keys())
    print(f"\n--- {cat} ---")
    print(f"{'Pair':<20} {'Diff':>8} {'CI_low':>8} {'CI_high':>8} {'p':>7}  ")
    print("-" * 55)

    for g1_name, g2_name in combinations(group_names, 2):
        diff, lo, hi, p = boot_quantile_diff(
            groups[g1_name], groups[g2_name], QUANTILE, N_BOOT
        )
        sig = " *" if p < ALPHA else ""
        try:
            print(f"{g1_name+' vs '+g2_name:<20} {diff:>8.3f} {lo:>8.3f} {hi:>8.3f} {p:>7.4f}{sig}")
        except:
            print(f"{str(g1_name)+' vs '+str(g2_name):<20} {diff:>8.3f} {lo:>8.3f} {hi:>8.3f} {p:>7.4f}{sig}")
'''
;

''

In [6]:
# Make a separate table for only extreme precipitation events for the next testing.
precip_extreme = precip[precip['precipitation'] >= precip.groupby(cat_cols)['precipitation'].transform('quantile', 0.975)].dropna()

In [7]:
# ================================================================
# 4. BOOTSTRAP PAIRWISE COMPARISON — per categorical variable
#    Analogous to Tukey's HSD: which specific pairs of groups
#    have significantly different means?
#    Uses bootstrap CIs on the difference in means.
#    This is for use after keeping only values at least 97.5 percentile
#    per categorical combination; no distribution assumed.
# ================================================================
print("\n" + "=" * 60)
print("BOOTSTRAP PAIRWISE COMPARISON on extreme means (w/ Bonferroni correction applied)")
print("=" * 60)

def boot_mean_diff(g1, g2, n_boot, n_tests, alpha=0.05):
    """Bootstrap CI for the difference in mean between two groups non-normally distributed."""
    diffs = []
    for _ in range(n_boot):
        b1 = np.random.choice(g1, size=len(g1), replace=True)
        b2 = np.random.choice(g2, size=len(g2), replace=True)
        diffs.append(np.mean(b1) - np.mean(b2))
    diffs = np.array(diffs)
    reject = alpha/n_tests
    ci_lo, ci_hi = np.percentile(diffs, [reject/2*100, (1-reject/2)*100])
    # p-value: proportion of bootstrap diffs on the opposite side of 0
    p = 2 * min(np.mean(diffs >= 0), np.mean(diffs <= 0))
    return np.mean(g1) - np.mean(g2), ci_lo, ci_hi, p

for cat in cat_cols:
    groups     = {name: g['precipitation'].values for name, g in precip_extreme.groupby(cat)}
    group_names = list(groups.keys())
    print(f"\n--- {cat} ---")
    print(f"{'Pair':<20} {'Diff':>8} {'CI_low':>8} {'CI_high':>8} {'p':>7}  ")
    print("-" * 55)

    for g1_name, g2_name in combinations(group_names, 2):
        diff, lo, hi, p = boot_mean_diff(
            groups[g1_name], groups[g2_name], 9999, 24
        )
        sig = " *" if p < ALPHA else ""
        print(f"{g1_name+' vs '+g2_name:<20} {diff:>8.3f} {lo:>8.3f} {hi:>8.3f} {p:>7.4f}{sig}")


BOOTSTRAP PAIRWISE COMPARISON on extreme means (w/ Bonferroni correction applied)

--- era ---
Pair                     Diff   CI_low  CI_high       p  
-------------------------------------------------------
end-of-century vs historical    5.786    4.211    7.391  0.0000 *
end-of-century vs near-term    3.815    2.241    5.410  0.0000 *
end-of-century vs obs    5.786    4.279    7.302  0.0000 *
historical vs near-term   -1.971   -3.531   -0.406  0.0000 *
historical vs obs       0.000   -1.468    1.464  0.9975
near-term vs obs        1.971    0.447    3.567  0.0002 *

--- ARDT ---
Pair                     Diff   CI_low  CI_high       p  
-------------------------------------------------------
G18 vs TE               4.591    3.176    6.046  0.0000 *
G18 vs tARget           1.988    0.725    3.259  0.0000 *
TE vs tARget           -2.603   -4.111   -1.115  0.0000 *

--- model ---
Pair                     Diff   CI_low  CI_high       p  
--------------------------------------------------

In [8]:
############### Below is the loop to generate stats as described above for all watersheds ###################

for wshed in wshed_list:

    precip = pd.read_csv(f'intermediate_numpy_files/precip_for_sigtest_{wshed}.csv')
    print(f"===================================")
    print(f"Statistics for watershed {wshed}:")
    print(f"===================================\n")
    model = ols('precipitation ~ C(era) + C(ARDT) + C(model)', data=precip).fit()
    anova_result = sm.stats.anova_lm(model, type=2)
    print(anova_result)
    print("\n" + "=" * 55)
    print("ONE-WAY ANOVA (per categorical variable)")
    print("=" * 55)
    cat_cols=['era', 'ARDT', 'model']
    for cat in cat_cols:
        groups = [group['precipitation'].values for _, group in precip.groupby(cat)]
        f_stat, p_val = stats.f_oneway(*groups)
        print(f"\n{cat}:  F = {f_stat:.4f},  p = {p_val:.4f}",
              "<-- significant" if p_val < 0.05 else "")
    
    print("\n" + "=" * 55)
    print("TUKEY HSD (pairwise group comparisons per variable)")
    print("=" * 55)
    
    for cat in cat_cols:
        print(f"\n--- {cat} ---")
        tukey = pairwise_tukeyhsd(
            endog  = precip['precipitation'],
            groups = precip[cat],
            alpha  = 0.05,
        )
        print(tukey)
    
    print("\n" + "=" * 55)
    print("VARIANCE TESTS (per categorical variable)")
    print("=" * 55)
    
    for cat in cat_cols:
        groups = [group['precipitation'].values for _, group in precip.groupby(cat)]
    
        lev_stat,  lev_p  = stats.levene(*groups)   # robust to non-normality
        bart_stat, bart_p = stats.bartlett(*groups)  # assumes normality
    
        print(f"\n{cat}:")
        print(f"  Levene:   W = {lev_stat:.4f},  p = {lev_p:.4f}",
              "<-- unequal variances" if lev_p < 0.05 else "")
        print(f"  Bartlett: T = {bart_stat:.4f},  p = {bart_p:.4f}",
              "<-- unequal variances" if bart_p < 0.05 else "")
    QUANTILE  = 0.99   # <-- change to 0.99 etc.
    N_BOOT    = 2000
    N_PERM    = 2000
    ALPHA     = 0.05
    precip_extreme = precip[precip['precipitation'] >= precip.groupby(cat_cols)['precipitation'].transform('quantile', 0.975)].dropna()
    print("\n" + "=" * 60)
    print("BOOTSTRAP PAIRWISE COMPARISON on extreme means (w/ Bonferroni correction applied)")
    print("=" * 60)
    
    def boot_mean_diff(g1, g2, n_boot, n_tests, alpha=0.05):
        """Bootstrap CI for the difference in mean between two groups non-normally distributed."""
        diffs = []
        for _ in range(n_boot):
            b1 = np.random.choice(g1, size=len(g1), replace=True)
            b2 = np.random.choice(g2, size=len(g2), replace=True)
            diffs.append(np.mean(b1) - np.mean(b2))
        diffs = np.array(diffs)
        reject = alpha/n_tests
        ci_lo, ci_hi = np.percentile(diffs, [reject/2*100, (1-reject/2)*100])
        # p-value: proportion of bootstrap diffs on the opposite side of 0
        p = 2 * min(np.mean(diffs >= 0), np.mean(diffs <= 0))
        return np.mean(g1) - np.mean(g2), ci_lo, ci_hi, p
    
    for cat in cat_cols:
        groups     = {name: g['precipitation'].values for name, g in precip_extreme.groupby(cat)}
        group_names = list(groups.keys())
        print(f"\n--- {cat} ---")
        print(f"{'Pair':<20} {'Diff':>8} {'CI_low':>8} {'CI_high':>8} {'p':>7}  ")
        print("-" * 55)
    
        for g1_name, g2_name in combinations(group_names, 2):
            diff, lo, hi, p = boot_mean_diff(
                groups[g1_name], groups[g2_name], 9999, 24
            )
            sig = " *" if p < ALPHA else ""
            print(f"{g1_name+' vs '+g2_name:<20} {diff:>8.3f} {lo:>8.3f} {hi:>8.3f} {p:>7.4f}{sig}")


Statistics for watershed 1503:

                df        sum_sq       mean_sq           F    PR(>F)
C(era)         3.0  6.266731e+02    208.891030    3.243772  0.020997
C(ARDT)        2.0  1.108581e+05  55429.029656  860.731826  0.000000
C(model)       4.0  1.020184e+05  25504.592508  396.049049  0.000000
Residual  118297.0  7.618038e+06     64.397560         NaN       NaN

ONE-WAY ANOVA (per categorical variable)

era:  F = 3.1558,  p = 0.0237 <-- significant

ARDT:  F = 845.6557,  p = 0.0000 <-- significant

model:  F = 381.5594,  p = 0.0000 <-- significant

TUKEY HSD (pairwise group comparisons per variable)

--- era ---
      Multiple Comparison of Means - Tukey HSD, FWER=0.05      
    group1       group2   meandiff p-adj   lower  upper  reject
---------------------------------------------------------------
end-of-century historical   0.0813 0.6035 -0.0876 0.2501  False
end-of-century  near-term   0.1971 0.0114  0.0324 0.3618   True
end-of-century        obs   0.0813 0.6035 -0.08